## Loading and exploring the IMDb dataset

I load the IMDb dataset and explore its contents, labels, and review text.

In [25]:
from datasets import load_dataset
from IPython.display import display

ds = load_dataset("stanfordnlp/imdb") # downloading the dataset from huggingface
ds.save_to_disk("data/imdb") # saves the dataset to the project folder

Saving the dataset (1/1 shards): 100%|██████████| 50000/50000 [00:00<00:00, 214536.04 examples/s]


In [26]:
display(ds)

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 25000
    })
    unsupervised: Dataset({
        features: ['text', 'label'],
        num_rows: 50000
    })
})

In [27]:
df = ds["train"].to_pandas()
display(df.head(10))

,text,label
0,"I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered ""controversial"" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and married men.<br /><br />What kills me about I AM CURIOUS-YELLOW is that 40 years ago, this was considered pornographic. Really, the sex and nudity scenes are few and far between, even then it's not shot like some cheaply made porno. While my countrymen mind find it shocking, in reality sex and nudity are a major staple in Swedish cinema. Even Ingmar Bergman, arguably their answer to good old boy John Ford, had sex scenes in his films.<br /><br />I do commend the filmmakers for the fact that any sex shown in the film is shown for artistic purposes rather than just to shock people and make money to be shown in pornographic theaters in America. I AM CURIOUS-YELLOW is a good film for anyone wanting to study the meat and potatoes (no pun intended) of Swedish cinema. But really, this film doesn't have much of a plot.",0
1,"""I Am Curious: Yellow"" is a risible and pretentious steaming pile. It doesn't matter what one's political views are because this film can hardly be taken seriously on any level. As for the claim that frontal male nudity is an automatic NC-17, that isn't true. I've seen R-rated films with male nudity. Granted, they only offer some fleeting views, but where are the R-rated films with gaping vulvas and flapping labia? Nowhere, because they don't exist. The same goes for those crappy cable shows: schlongs swinging in the breeze but not a clitoris in sight. And those pretentious indie movies like The Brown Bunny, in which we're treated to the site of Vincent Gallo's throbbing johnson, but not a trace of pink visible on Chloe Sevigny. Before crying (or implying) ""double-standard"" in matters of nudity, the mentally obtuse should take into account one unavoidably obvious anatomical difference between men and women: there are no genitals on display when actresses appears nude, and the same cannot be said for a man. In fact, you generally won't see female genitals in an American film in anything short of porn or explicit erotica. This alleged double-standard is less a double standard than an admittedly depressing ability to come to terms culturally with the insides of women's bodies.",0
2,"If only to avoid making this type of film in the future. This film is interesting as an experiment but tells no cogent story.<br /><br />One might feel virtuous for sitting thru it because it touches on so many IMPORTANT issues but it does so without any discernable motive. The viewer comes away with no new perspectives (unless one comes up with one while one's mind wanders, as it will invariably do during this pointless film).<br /><br />One might better spend one's time staring out a window at a tree growing.<br /><br />",0
3,"This film was probably inspired by Godard's Masculin, féminin and I urge you to see that film instead.<br /><br />The film has two strong elements and those are, (1) the realistic acting (2) the impressive, undeservedly good, photo. Apart from that, what strikes me most is the endless stream of silliness. Lena Nyman has to be most annoying actress in the world. She acts so stupid and with all the nudity in this film,...it's unattractive. Comparing to Godard's film, intellectuality has been r

In [28]:
df.columns # The dataset has two columns

Index(['text', 'label'], dtype='object')

In [29]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 25000 entries, 0 to 24999
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    25000 non-null  object
 1   label   25000 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 390.8+ KB


In [30]:
df["label"].nunique() # We have to values in the Label column
df["label"].unique() # The two values are 0 and 1

array([0, 1])

In [31]:
df["label"].value_counts()

label
0    12500
1    12500
Name: count, dtype: int64

I found that the training split contains equal numbers of positive and negative reviews: 12,500 of each. The labels are 0 and 1, respectively.

In [ ]:
import pandas as pd

display(df.head(10))
display(df.groupby("label").size().rename("review_count"))

In [ ]:
display(df.loc[df["label"] == 0, ["text", "label"]].head(5))

### Interpreting the sentiment labels

From the review examples, I interpret label 0 as negative sentiment and label 1 as positive sentiment.

In [ ]:
df["length"] = df["text"].str.len()  # Length in characters
print(df["length"].describe())

In [ ]:
df["word_count"] = df["clean_text"].str.split().str.len()
print(df["word_count"].describe())

In [36]:
df[["text", "label"]].isna().sum() # The dataset card reports no missing values, but I check anyway.

text     0
label    0
dtype: int64

The Hugging Face dataset card lists the reviews as English. I treat the dataset as English for this project rather than running language detection on every review.

In [37]:
html_tag_pattern = r"<[^>]+>"
has_html_tags = df["text"].str.contains(html_tag_pattern, regex=True, na=False)
print(f"Reviews containing HTML tags: {has_html_tags.sum()}")
display(df.loc[has_html_tags, "text"].head())

Reviews containing HTML tags: 14667


0                                                                                                                                                                                  I rented I AM CURIOUS-YELLOW from my video store because of all the controversy that surrounded it when it was first released in 1967. I also heard that at first it was seized by U.S. customs if it ever tried to enter this country, therefore being a fan of films considered "controversial" I really had to see this for myself.<br /><br />The plot is centered around a young Swedish drama student named Lena who wants to learn everything she can about life. In particular she wants to focus her attentions to making some sort of documentary on what the average Swede thought about certain political issues such as the Vietnam War and race issues in the United States. In between asking politicians and ordinary denizens of Stockholm about their opinions on politics, she has sex with her drama teacher, classmates, and mar

In [38]:
import html
import re

def clean_html(text):
    text = html.unescape(text) # Decodes for example &amp;
    text = re.sub(r"<[^>]+>", " ", text) # Replaces HTML tags with spaces
    text = re.sub(r"\s+", " ", text).strip()  # Removes extra spaces
    return text

df["clean_text"] = df["text"].apply(clean_html)

## Looking for duplicates

In [39]:
df["text"].duplicated().sum() # Looking for duplicated reviews

96

I found 96 duplicate review rows in the training split.

## Removing duplicates

In [ ]:
duplicate_summary = (
    df.groupby("text", as_index=False)
    .agg(
        occurrences=("label", "size"),
        label_count=("label", "nunique"),
        labels=("label", lambda values: sorted(values.unique()))
    )
    .query("occurrences > 1")
)

duplicate_summary["text_preview"] = duplicate_summary["text"].str[:120]
display(duplicate_summary[["text_preview", "occurrences", "label_count", "labels"]])

# I remove duplicate reviews only when their labels agree.
safe_duplicate_texts = duplicate_summary.loc[
    duplicate_summary["label_count"] == 1, "text"
]
rows_to_remove = (
    df["text"].duplicated(keep="first")
    & df["text"].isin(safe_duplicate_texts)
)

conflicting_duplicates = duplicate_summary[duplicate_summary["label_count"] > 1]
print(f"Duplicate text groups: {len(duplicate_summary)}")
print(f"Duplicate rows safe to remove: {rows_to_remove.sum()}")
print(f"Groups with conflicting labels (kept for review): {len(conflicting_duplicates)}")

display(df.loc[rows_to_remove, ["label"]].head(10))

# I keep the original df unchanged and use df_clean for later analysis and modeling.
df_clean = df.loc[~rows_to_remove].reset_index(drop=True)
print(f"Rows before: {len(df)}; rows after: {len(df_clean)}")

In [41]:
df["word_count"].describe()

count    25000.000000
mean       233.787200
std        173.733032
min         10.000000
25%        127.000000
50%        174.000000
75%        284.000000
max       2470.000000
Name: word_count, dtype: float64

### I found 25,000 reviews in the training dataset.

I found that the shortest review contains 10 words and the longest contains 2,470. The average review contains approximately 234 words.

- **Median:** 174 words — half of the reviews contain fewer words, and half contain more.
- **25th percentile:** 127 words — 25% of reviews contain 127 words or fewer.
- **75th percentile:** 284 words — 75% of reviews contain 284 words or fewer.

I noticed that the mean is higher than the median, suggesting that a few very long reviews pull the average upward.

In [42]:
len(df_clean)

24904

In [43]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 24904 entries, 0 to 24903
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype 
---  ------      --------------  ----- 
 0   text        24904 non-null  object
 1   label       24904 non-null  int64 
 2   length      24904 non-null  int64 
 3   word_count  24904 non-null  int64 
 4   clean_text  24904 non-null  object
dtypes: int64(3), object(2)
memory usage: 972.9+ KB


## Transforming the text reviews

### I use TF-IDF to turn the reviews into numbers that a model can use. It assigns each word a weight in every review. For example, the word movie often gets a smaller weight because it appears in many reviews, while a less common word that expresses an opinion may get a higher weight.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

X_train_text, X_test_text, y_train, y_test = train_test_split(
    df_clean["clean_text"],
    df_clean["label"],
    test_size=0.2,
    random_state=42,
    stratify=df_clean["label"]
)

vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print("Number of reviews:", len(df_clean))
print("TF-IDF shape for training:", X_train.shape)
print("TF-IDF shape for development holdout:", X_test.shape)

### Training split (19,923, 68,340): I represent 19,923 reviews using 68,340 terms learned from the training data.

### Development holdout (4,981, 68,340): I represent 4,981 reviews using the same 68,340 terms.

In [45]:
row_number = 0
row = X_train[row_number]

terms = vectorizer.get_feature_names_out()

review_vector = pd.DataFrame({
    "word": terms[row.indices],
    "tfidf_weight": row.data
}).sort_values("tfidf_weight", ascending=False)

print(X_train_text.iloc[row_number])  # The cleaned review
display(review_vector.head())       # Terms and TF-IDF weights

Disappointing film. Performance of actors is weak. Sets are fine, could have been better. The story is also weak. Battle sequences are awful. Sounds and quality of film are trashy. The history of Kazakh people was told very poorly. This film should have included more Kazakh actors, in leading roles. And also should have been in Kazakh language. Kuno Bekker and Jay Hernandez are Hispanic origins. I don't get it. Since when Hispanic people play Turkic-Mongolian people. This film is shame of Kazakh cinema. Rustam Ibragimbekov disappointed me. He is one of the finest filmmakers in the world. Czekh director is excused, since he is not nomadic origin, he cannot know true spirit and history of nomads.


,word,tfidf_weight
25,kazakh,0.606555
43,hispanic,0.238933
64,czekh,0.157912
40,bekker,0.157912
56,ibragimbekov,0.157912


## Training the baseline model

In [46]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

baseline_model = LogisticRegression(max_iter=1000)
baseline_model.fit(X_train, y_train)

baseline_predictions = baseline_model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, baseline_predictions))
print("Classification Report:")
print(classification_report(y_test, baseline_predictions))

Accuracy: 0.8787392089941779
Classification Report:
              precision    recall  f1-score   support

           0       0.89      0.87      0.88      2486
           1       0.87      0.89      0.88      2495

    accuracy                           0.88      4981
   macro avg       0.88      0.88      0.88      4981
weighted avg       0.88      0.88      0.88      4981



## Building and training an artificial neural network (ANN)

I use Keras with TensorFlow as its backend.

In [47]:
import os
os.environ["SETUPTOOLS_USE_DISTUTILS"] = "local"
os.environ["KERAS_BACKEND"] = "tensorflow"

import setuptools
import distutils

In [48]:
import numpy as np  # Import NumPy as np for numerical arrays and data types.
import keras  # Import Keras to build and train the neural network.
from keras import Sequential  # Import a model that stacks layers in sequence.
from keras.layers import Dense, Input  # Import the input layer and fully connected Dense layers.
from sklearn.feature_extraction.text import TfidfVectorizer  # Import the text-to-numeric TF-IDF vectorizer.

# I limit this neural network to 1,000 useful terms to reduce memory use.
ann_vectorizer = TfidfVectorizer(max_features=1000)
# I learn the vocabulary from training text, calculate TF-IDF values, use 32-bit numbers, and create a dense array for Keras.
X_train_ann = ann_vectorizer.fit_transform(X_train_text).astype(np.float32).toarray()
# I transform holdout text with the training vocabulary without fitting the vectorizer again.
X_test_ann = ann_vectorizer.transform(X_test_text).astype(np.float32).toarray()

# I convert labels to integer NumPy arrays: 0 = negative and 1 = positive.
y_train_ann = y_train.to_numpy(dtype=np.int64)
y_test_ann = y_test.to_numpy(dtype=np.int64)

# I set a random seed to make model initialization and training more reproducible.
keras.utils.set_random_seed(42)
# I build the neural network by stacking input, hidden, and output layers.
ann_model = Sequential([
    # I represent each review with one value for each TF-IDF vocabulary term.
    Input(shape=(X_train_ann.shape[1],)),
    # I use 32 neurons in the hidden layer, with ReLU to learn non-linear patterns.
    Dense(32, activation="relu"),
    # I use two output probabilities, one for each sentiment class.
    Dense(2, activation="softmax"),
])

# I configure how the model learns and how I measure its performance.
ann_model.compile(
    # I use Adam to adjust the model weights and reduce prediction error.
    optimizer="adam",
    # I use this loss because the two classes are integer IDs (0 and 1).
    loss="sparse_categorical_crossentropy",
    # I track the fraction of correct predictions.
    metrics=["accuracy"],
)

# I train the network on the training features and their labels.
history = ann_model.fit(
    X_train_ann,  # TF-IDF features used as model input.
    y_train_ann,  # Correct sentiment labels used as training targets.
    validation_split=0.1,  # Keep 10% of training data for validation during training.
    epochs=5,  # Make five passes through the training data.
    batch_size=64,  # Update weights after processing batches of 64 reviews.
)

# I evaluate on the development holdout; verbose=0 hides the detailed progress display.
ann_loss, ann_accuracy = ann_model.evaluate(X_test_ann, y_test_ann, verbose=0)
# I print the development-holdout accuracy to six decimal places.
print(f"ANN development holdout accuracy: {ann_accuracy:.6f}")

Epoch 1/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 3ms/step - accuracy: 0.7282 - loss: 0.5888 - val_accuracy: 0.8490 - val_loss: 0.3609
Epoch 2/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8604 - loss: 0.3319 - val_accuracy: 0.8555 - val_loss: 0.3344
Epoch 3/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8718 - loss: 0.3019 - val_accuracy: 0.8595 - val_loss: 0.3346
Epoch 4/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8759 - loss: 0.2925 - val_accuracy: 0.8565 - val_loss: 0.3378
Epoch 5/5
281/281 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8781 - loss: 0.2878 - val_accuracy: 0.8555 - val_loss: 0.3409
ANN development holdout accuracy: 0.854447


My first ANN used **1,000 TF-IDF features** without regularization and achieved **85.44% accuracy** on the 4,981-review holdout split.

My second ANN used **5,000 features**, L2 regularization, and early stopping, and achieved **88.46% accuracy** on that split. I observed that validation accuracy peaked at **89.86% in epoch 2**; early stopping restored the weights with the best validation loss.

For comparison, my Logistic Regression model with the full 68,340-feature vocabulary achieved **87.87%**, and my LinearSVC model with the same vocabulary achieved **89.16%** on the holdout split. Logistic Regression using the same **5,000 features** as my second ANN achieved **87.61%**. I use the holdout results to compare models and reserve the separate official IMDb test split for my final evaluation.

In [49]:
# I limit this neural network to 5,000 useful terms.
ann_vectorizer = TfidfVectorizer(max_features=5000)
# I learn the vocabulary from training text, calculate TF-IDF values, use 32-bit numbers, and create a dense array for Keras.
X_train_ann = ann_vectorizer.fit_transform(X_train_text).astype(np.float32).toarray()
# I transform holdout text with the training vocabulary without fitting the vectorizer again.
X_test_ann = ann_vectorizer.transform(X_test_text).astype(np.float32).toarray()

# I convert labels to integer NumPy arrays: 0 = negative and 1 = positive.
y_train_ann = y_train.to_numpy(dtype=np.int64)
y_test_ann = y_test.to_numpy(dtype=np.int64)

# I set a random seed to make model initialization and training more reproducible.
keras.utils.set_random_seed(42)
# I build the neural network by stacking input, hidden, and output layers.
ann_model = Sequential([
    # I represent each review with one value for each TF-IDF vocabulary term.
    Input(shape=(X_train_ann.shape[1],)),
    Dense(
    32,
    activation="relu",
    kernel_regularizer=keras.regularizers.l2(1e-4)
),
    # I use two output probabilities, one for each sentiment class.
    Dense(2, activation="softmax"),
])

# I configure how the model learns and how I measure its performance.
ann_model.compile(
    # I use Adam to adjust the model weights and reduce prediction error.
    optimizer="adam",
    # I use this loss because the two classes are integer IDs (0 and 1).
    loss="sparse_categorical_crossentropy",
    # I track the fraction of correct predictions.
    metrics=["accuracy"],
)

# I train the network on the training features and their labels.
history = ann_model.fit(
    X_train_ann,  # TF-IDF features used as model input.
    y_train_ann,  # Correct sentiment labels used as training targets.
    validation_split=0.1,  # Keep 10% of training data for validation during training.
    epochs=20,  # Make 20 passes through the training data.
    batch_size=64,  # Update weights after processing batches of 64 reviews.
    callbacks=[
        keras.callbacks.EarlyStopping(
            monitor="val_loss",
            patience=2,
            restore_best_weights=True
        )
    ]
)

# I evaluate on the development holdout; verbose=0 hides the detailed progress display.
ann_loss, ann_accuracy = ann_model.evaluate(X_test_ann, y_test_ann, verbose=0)
# I print the development-holdout accuracy to six decimal places.
print(f"ANN development holdout accuracy: {ann_accuracy:.6f}")

Epoch 1/20
281/281 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.7624 - loss: 0.5747 - val_accuracy: 0.8851 - val_loss: 0.3306
Epoch 2/20
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.9011 - loss: 0.2941 - val_accuracy: 0.8986 - val_loss: 0.3029
Epoch 3/20
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9201 - loss: 0.2518 - val_accuracy: 0.8946 - val_loss: 0.3050
Epoch 4/20
281/281 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.9296 - loss: 0.2335 - val_accuracy: 0.8936 - val_loss: 0.3121
ANN development holdout accuracy: 0.884561


### Evaluating Logistic Regression with 5,000 features

In [50]:
logistic_5000 = LogisticRegression(max_iter=1000)

logistic_5000.fit(X_train_ann, y_train_ann)

logistic_5000_predictions = logistic_5000.predict(X_test_ann)

print("Accuracy:", accuracy_score(y_test_ann, logistic_5000_predictions))
print("Classification Report:")
print(classification_report(y_test_ann, logistic_5000_predictions))

Accuracy: 0.8761292913069665
Classification Report:
              precision    recall  f1-score   support

           0       0.88      0.86      0.87      2486
           1       0.87      0.89      0.88      2495

    accuracy                           0.88      4981
   macro avg       0.88      0.88      0.88      4981
weighted avg       0.88      0.88      0.88      4981



My Logistic Regression model using the same **5,000 TF-IDF features** as the regularized ANN achieved **87.61% accuracy** on the 4,981-review holdout split (about 617 incorrect predictions). I found that its class-wise precision, recall, and F1 scores are balanced. My regularized ANN scored 88.46% on this split. I have not tested whether this difference is statistically significant.

## Model comparison

I evaluated all the models below on the same stratified 20% holdout split from IMDb's training set (4,981 reviews).

- **Logistic Regression baseline:** 68,340 TF-IDF features; accuracy: 87.87%.
- **ANN, no regularization:** 1,000 TF-IDF features; accuracy: 85.44%.
- **ANN, L2 regularization and early stopping:** 5,000 TF-IDF features; accuracy: 88.46%.
- **Logistic Regression:** 5,000 TF-IDF features; accuracy: 87.61%.
- **LinearSVC:** 68,340 TF-IDF features; accuracy: **89.16%**.

My LinearSVC model had the highest accuracy on this holdout split. My regularized ANN also outperformed both Logistic Regression runs, although it used fewer features than the full-vocabulary models. I used the holdout split to compare models, so I kept IMDb's official test split separate for my final evaluation. This helps me estimate how well my selected model performs on unseen reviews.

## Testing LinearSVC on the development holdout split

In [51]:
from sklearn.svm import LinearSVC
from sklearn.metrics import accuracy_score, classification_report

svm_model = LinearSVC(random_state=42)
svm_model.fit(X_train, y_train)

svm_predictions = svm_model.predict(X_test)

print(f"Accuracy: {accuracy_score(y_test, svm_predictions):.6f}")
print(classification_report(y_test, svm_predictions))

Accuracy: 0.891588
              precision    recall  f1-score   support

           0       0.90      0.88      0.89      2486
           1       0.89      0.90      0.89      2495

    accuracy                           0.89      4981
   macro avg       0.89      0.89      0.89      4981
weighted avg       0.89      0.89      0.89      4981



## Final evaluation on IMDb's official test split

After comparing my models on the development holdout split, I evaluated my selected full-vocabulary LinearSVC on IMDb's separate official test split. It achieved **87.608% accuracy** on 25,000 reviews. I kept this test split separate from model selection so it could provide a final evaluation on data I had not used for tuning.

In [ ]:
official_test_df = ds["test"].to_pandas()
official_test_df["clean_text"] = official_test_df["text"].apply(clean_html)

X_official_test = vectorizer.transform(official_test_df["clean_text"])
y_official_test = official_test_df["label"].to_numpy()
official_svm_predictions = svm_model.predict(X_official_test)

official_test_accuracy = accuracy_score(y_official_test, official_svm_predictions)
print(f"Official IMDb test accuracy: {official_test_accuracy:.6f}")
print(classification_report(y_official_test, official_svm_predictions, target_names=["negative", "positive"]))

## Reviewing misclassified official-test reviews

In the following cell, I display a small sample of reviews the model misclassified in the official test split so I can inspect them.

In [ ]:
error_mask = official_svm_predictions != y_official_test
misclassified = official_test_df.loc[error_mask, ["text", "label"]].copy()
misclassified["predicted_label"] = official_svm_predictions[error_mask]
misclassified["actual_sentiment"] = misclassified["label"].map({0: "negative", 1: "positive"})
misclassified["predicted_sentiment"] = misclassified["predicted_label"].map({0: "negative", 1: "positive"})
misclassified["text_preview"] = misclassified["text"].str.slice(0, 500)

print(f"Misclassified official-test reviews: {error_mask.sum()} of {len(official_test_df)}")
display(misclassified[["actual_sentiment", "predicted_sentiment", "text_preview"]].head(10))